In [ ]:
#!/usr/bin/env python3
"""
Interactive Image Watermarking System with MPI + CUDA
COMPLETE FIXED VERSION - Ready for Google Colab
"""

import os
import subprocess
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
import io

# ============================================================================
# STEP 1: SETUP ENVIRONMENT
# ============================================================================

def setup_environment():
    """Install all dependencies"""
    print("="*60)
    print("🔧 SETTING UP ENVIRONMENT")
    print("="*60)

    print("\n📊 GPU Information:")
    os.system("nvidia-smi --query-gpu=name,memory.total --format=csv")

    print("\n📦 Installing dependencies...")
    os.system("apt-get update -qq && apt-get install -y -qq build-essential libopenmpi-dev openmpi-bin libpng-dev libjpeg-dev wget > /dev/null 2>&1")
    os.system("pip install -q numpy matplotlib pillow pandas")

    for d in ['src', 'include', 'build', 'bin', 'data', 'results', 'uploads']:
        os. makedirs(d, exist_ok=True)

    print("\n📥 Downloading headers...")
    os.system("wget -q -O include/stb_image.h https://raw.githubusercontent.com/nothings/stb/master/stb_image.h")
    os.system("wget -q -O include/stb_image_write.h https://raw.githubusercontent.com/nothings/stb/master/stb_image_write.h")

    print("✅ Setup complete!\n")

# ============================================================================
# STEP 2: CREATE SOURCE FILES
# ============================================================================

def create_source_files():
    """Generate all source code files"""
    print("="*60)
    print("💾 CREATING SOURCE FILES")
    print("="*60)

    with open('include/watermark.h', 'w') as f:
        f.write("""#ifndef WATERMARK_H
#define WATERMARK_H

#include <stdint.h>
#include <stdbool.h>

typedef struct {
    int width;
    int height;
    int channels;
    unsigned char *data;
} Image;

typedef struct {
    int width;
    int height;
    unsigned char *data;
} Watermark;

typedef struct {
    int alpha;
    bool use_gpu;
} Config;

#ifdef __cplusplus
extern "C" {
#endif

Image* load_image(const char *filename);
void save_image(const char *filename, Image *img);
void free_image(Image *img);
Watermark* load_logo(const char *filename, int target_width, int target_height);
void free_watermark(Watermark *wm);
void embed_watermark_cuda(Image *img, Watermark *wm, Config *config);
Watermark* extract_watermark_cuda(Image *img, Config *config);
void embed_watermark_cpu(Image *img, Watermark *wm, Config *config);
Watermark* extract_watermark_cpu(Image *img, Config *config);
double calculate_similarity(Watermark *wm1, Watermark *wm2);
double calculate_image_similarity(Image *img1, Image *img2);
double calculate_psnr(Image *img1, Image *img2);

#ifdef __cplusplus
}
#endif

#endif
""")

    with open('src/image_utils.c', 'w') as f:
        f.write("""#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#define STB_IMAGE_IMPLEMENTATION
#define STB_IMAGE_WRITE_IMPLEMENTATION
#include "../include/stb_image.h"
#include "../include/stb_image_write.h"
#include "../include/watermark.h"

Image* load_image(const char *filename) {
    Image *img = (Image*)malloc(sizeof(Image));
    img->data = stbi_load(filename, &img->width, &img->height, &img->channels, 3);
    if (!img->data) {
        fprintf(stderr, "Error loading:  %s\\n", filename);
        free(img);
        return NULL;
    }
    img->channels = 3;
    printf("✓ Loaded:  %dx%d\\n", img->width, img->height);
    return img;
}

void save_image(const char *filename, Image *img) {
    stbi_write_png(filename, img->width, img->height, img->channels,
                   img->data, img->width * img->channels);
    printf("✓ Saved:  %s\\n", filename);
}

void free_image(Image *img) {
    if (img) {
        if (img->data) stbi_image_free(img->data);
        free(img);
    }
}

Watermark* load_logo(const char *filename, int target_width, int target_height) {
    int orig_width, orig_height, orig_channels;
    unsigned char *orig_data = stbi_load(filename, &orig_width, &orig_height, &orig_channels, 1);

    if (!orig_data) {
        fprintf(stderr, "Error loading logo: %s\\n", filename);
        return NULL;
    }

    Watermark *wm = (Watermark*)malloc(sizeof(Watermark));
    wm->width = target_width;
    wm->height = target_height;
    wm->data = (unsigned char*)malloc(target_width * target_height);

    for (int y = 0; y < target_height; y++) {
        for (int x = 0; x < target_width; x++) {
            int src_x = (x * orig_width) / target_width;
            int src_y = (y * orig_height) / target_height;
            unsigned char pixel = orig_data[src_y * orig_width + src_x];
            wm->data[y * target_width + x] = (pixel > 128) ? 255 : 0;
        }
    }
    stbi_image_free(orig_data);
    printf("✓ Logo loaded: %dx%d\\n", target_width, target_height);
    return wm;
}

void free_watermark(Watermark *wm) {
    if (wm) {
        if (wm->data) free(wm->data);
        free(wm);
    }
}

double calculate_similarity(Watermark *wm1, Watermark *wm2) {
    if (wm1->width != wm2->width || wm1->height != wm2->height) return 0.0;
    int matches = 0;
    int total = wm1->width * wm1->height;
    for (int i = 0; i < total; i++) {
        if (wm1->data[i] == wm2->data[i]) matches++;
    }
    return (double)matches / total * 100.0;
}

double calculate_image_similarity(Image *img1, Image *img2) {
    if (img1->width != img2->width || img1->height != img2->height) return 0.0;
    int size = img1->width * img1->height * img1->channels;
    long long sum_sq_diff = 0;

    for (int i = 0; i < size; i++) {
        int diff = (int)img1->data[i] - (int)img2->data[i];
        sum_sq_diff += diff * diff;
    }

    double mse = (double)sum_sq_diff / size;
    double similarity = 100.0 / (1.0 + sqrt(mse));
    return similarity;
}

double calculate_psnr(Image *img1, Image *img2) {
    if (img1->width != img2->width || img1->height != img2->height) return 0.0;
    int size = img1->width * img1->height * img1->channels;
    double mse = 0.0;
    for (int i = 0; i < size; i++) {
        double diff = (double)img1->data[i] - (double)img2->data[i];
        mse += diff * diff;
    }
    mse /= size;
    if (mse == 0) return INFINITY;
    return 10.0 * log10((255.0 * 255.0) / mse);
}
""")

    # SPATIAL DOMAIN CUDA - LSB EMBEDDING
    with open('src/watermark_cuda.cu', 'w') as f:
        f.write("""#include <cuda_runtime.h>
#include <stdio.h>
#include "../include/watermark.h"

#define BLOCK_SIZE 16
#define CUDA_CHECK(call) { \\
    cudaError_t err = call; \\
    if (err != cudaSuccess) { \\
        fprintf(stderr, "CUDA error: %s\\n", cudaGetErrorString(err)); \\
        exit(EXIT_FAILURE); \\
    } \\
}

__global__ void embed_lsb_kernel(unsigned char *img, unsigned char *wm,
                                  int img_width, int img_height,
                                  int wm_width, int wm_height, int alpha) {
    int x = blockIdx.x * blockDim.x + threadIdx. x;
    int y = blockIdx.y * blockDim. y + threadIdx.y;

    if (x >= img_width || y >= img_height) return;

    int wm_x = (x * wm_width) / img_width;
    int wm_y = (y * wm_height) / img_height;

    if (wm_x >= wm_width || wm_y >= wm_height) return;

    unsigned char wm_bit = wm[wm_y * wm_width + wm_x];
    int img_idx = (y * img_width + x) * 3;

    if (wm_bit > 128) {
        // White pixel in watermark - embed 1
        img[img_idx] |= 1;
        img[img_idx + 1] |= 1;
        img[img_idx + 2] |= 1;
    } else {
        // Black pixel in watermark - embed 0
        img[img_idx] &= 0xFE;
        img[img_idx + 1] &= 0xFE;
        img[img_idx + 2] &= 0xFE;
    }
}

__global__ void extract_lsb_kernel(unsigned char *img, unsigned char *wm,
                                     int img_width, int img_height,
                                     int wm_width, int wm_height) {
    int wm_x = blockIdx.x * blockDim. x + threadIdx.x;
    int wm_y = blockIdx.y * blockDim. y + threadIdx.y;

    if (wm_x >= wm_width || wm_y >= wm_height) return;

    int img_x = (wm_x * img_width) / wm_width;
    int img_y = (wm_y * img_height) / wm_height;

    int img_idx = (img_y * img_width + img_x) * 3;

    int bit_count = 0;
    bit_count += (img[img_idx] & 1);
    bit_count += (img[img_idx + 1] & 1);
    bit_count += (img[img_idx + 2] & 1);

    wm[wm_y * wm_width + wm_x] = (bit_count >= 2) ? 255 : 0;
}

void embed_watermark_cuda(Image *img, Watermark *wm, Config *cfg) {
    int img_size = img->width * img->height * 3;
    int wm_size = wm->width * wm->height;

    unsigned char *d_img, *d_wm;

    CUDA_CHECK(cudaMalloc(&d_img, img_size));
    CUDA_CHECK(cudaMalloc(&d_wm, wm_size));

    CUDA_CHECK(cudaMemcpy(d_img, img->data, img_size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_wm, wm->data, wm_size, cudaMemcpyHostToDevice));

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((img->width + BLOCK_SIZE - 1) / BLOCK_SIZE,
              (img->height + BLOCK_SIZE - 1) / BLOCK_SIZE);

    embed_lsb_kernel<<<grid, block>>>(d_img, d_wm, img->width, img->height,
                                       wm->width, wm->height, cfg->alpha);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(img->data, d_img, img_size, cudaMemcpyDeviceToHost));

    cudaFree(d_img);
    cudaFree(d_wm);
}

Watermark* extract_watermark_cuda(Image *img, Config *cfg) {
    int wm_width = img->width / 4;
    int wm_height = img->height / 4;

    Watermark *wm = (Watermark*)malloc(sizeof(Watermark));
    wm->width = wm_width;
    wm->height = wm_height;
    wm->data = (unsigned char*)malloc(wm_width * wm_height);

    int img_size = img->width * img->height * 3;
    int wm_size = wm_width * wm_height;

    unsigned char *d_img, *d_wm;

    CUDA_CHECK(cudaMalloc(&d_img, img_size));
    CUDA_CHECK(cudaMalloc(&d_wm, wm_size));

    CUDA_CHECK(cudaMemcpy(d_img, img->data, img_size, cudaMemcpyHostToDevice));

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((wm_width + BLOCK_SIZE - 1) / BLOCK_SIZE,
              (wm_height + BLOCK_SIZE - 1) / BLOCK_SIZE);

    extract_lsb_kernel<<<grid, block>>>(d_img, d_wm, img->width, img->height,
                                         wm_width, wm_height);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(wm->data, d_wm, wm_size, cudaMemcpyDeviceToHost));

    cudaFree(d_img);
    cudaFree(d_wm);

    return wm;
}
""")

    # SPATIAL DOMAIN CPU - LSB EMBEDDING
    with open('src/watermark_cpu.c', 'w') as f:
        f.write("""#include <stdio.h>
#include <stdlib.h>
#include "../include/watermark.h"

void embed_watermark_cpu(Image *img, Watermark *wm, Config *cfg) {
    for (int y = 0; y < img->height; y++) {
        for (int x = 0; x < img->width; x++) {
            int wm_x = (x * wm->width) / img->width;
            int wm_y = (y * wm->height) / img->height;

            unsigned char wm_bit = wm->data[wm_y * wm->width + wm_x];
            int img_idx = (y * img->width + x) * 3;

            if (wm_bit > 128) {
                img->data[img_idx] |= 1;
                img->data[img_idx + 1] |= 1;
                img->data[img_idx + 2] |= 1;
            } else {
                img->data[img_idx] &= 0xFE;
                img->data[img_idx + 1] &= 0xFE;
                img->data[img_idx + 2] &= 0xFE;
            }
        }
    }
}

Watermark* extract_watermark_cpu(Image *img, Config *cfg) {
    int wm_width = img->width / 4;
    int wm_height = img->height / 4;

    Watermark *wm = (Watermark*)malloc(sizeof(Watermark));
    wm->width = wm_width;
    wm->height = wm_height;
    wm->data = (unsigned char*)malloc(wm_width * wm_height);

    for (int wm_y = 0; wm_y < wm_height; wm_y++) {
        for (int wm_x = 0; wm_x < wm_width; wm_x++) {
            int img_x = (wm_x * img->width) / wm_width;
            int img_y = (wm_y * img->height) / wm_height;

            int img_idx = (img_y * img->width + img_x) * 3;

            int bit_count = 0;
            bit_count += (img->data[img_idx] & 1);
            bit_count += (img->data[img_idx + 1] & 1);
            bit_count += (img->data[img_idx + 2] & 1);

            wm->data[wm_y * wm_width + wm_x] = (bit_count >= 2) ? 255 : 0;
        }
    }

    return wm;
}
""")

    with open('src/main.c', 'w') as f:
        f.write("""#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include "../include/watermark.h"

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Usage: %s <mode> [args]\\n", argv[0]);
        return 1;
    }

    char *mode = argv[1];
    clock_t start = clock();

    if (strcmp(mode, "similarity") == 0) {
        if (argc < 4) {
            fprintf(stderr, "Usage: %s similarity <image1> <image2>\\n", argv[0]);
            return 1;
        }

        printf("\\n╔════════════════════════════════════╗\\n");
        printf("║   IMAGE SIMILARITY CHECK          ║\\n");
        printf("╚════════════════════════════════════╝\\n\\n");

        Image *img1 = load_image(argv[2]);
        Image *img2 = load_image(argv[3]);

        if (! img1 || !img2) {
            fprintf(stderr, "Error loading images\\n");
            return 1;
        }

        double similarity = calculate_image_similarity(img1, img2);
        double psnr = calculate_psnr(img1, img2);

        printf("\\n📊 RESULTS:\\n");
        printf("   Similarity: %.2f%%%%\\n", similarity);
        printf("   PSNR:  %. 2f dB\\n", psnr);

        if (similarity > 95.0) {
            printf("   ✅ VERY SIMILAR\\n");
        } else if (similarity > 80.0) {
            printf("   ✅ SIMILAR\\n");
        } else if (similarity > 60.0) {
            printf("   ⚠️  SOMEWHAT SIMILAR\\n");
        } else {
            printf("   ❌ DIFFERENT\\n");
        }

        free_image(img1);
        free_image(img2);

    } else if (strcmp(mode, "embed") == 0) {
        if (argc < 5) {
            fprintf(stderr, "Usage:  %s embed <image> <logo> <gpu|cpu>\\n", argv[0]);
            return 1;
        }

        printf("\\n╔════════════════════════════════════╗\\n");
        printf("║   EMBED WATERMARK                 ║\\n");
        printf("╚════════════════════════════════════╝\\n\\n");

        Image *img = load_image(argv[2]);
        if (!img) return 1;

        int wm_size = img->width / 4;
        Watermark *logo = load_logo(argv[3], wm_size, wm_size);
        if (!logo) {
            free_image(img);
            return 1;
        }

        printf("   Watermark size: %dx%d\\n", wm_size, wm_size);

        bool use_gpu = (strcmp(argv[4], "gpu") == 0);
        Config cfg;
        cfg.alpha = 1;
        cfg.use_gpu = use_gpu;

        printf("\\n🔧 Processing with %s... \\n", use_gpu ? "GPU" : "CPU");

        if (use_gpu) {
            embed_watermark_cuda(img, logo, &cfg);
        } else {
            embed_watermark_cpu(img, logo, &cfg);
        }

        save_image("results/watermarked_image.png", img);
        printf("\\n✅ Success! \\n");

        free_image(img);
        free_watermark(logo);

    } else if (strcmp(mode, "extract") == 0) {
        if (argc < 4) {
            fprintf(stderr, "Usage: %s extract <watermarked_image> <gpu|cpu>\\n", argv[0]);
            return 1;
        }

        printf("\\n╔════════════════════════════════════╗\\n");
        printf("║   EXTRACT WATERMARK               ║\\n");
        printf("╚════════════════════════════════════╝\\n\\n");

        Image *img = load_image(argv[2]);
        if (!img) return 1;

        bool use_gpu = (strcmp(argv[3], "gpu") == 0);
        Config cfg;
        cfg.alpha = 1;
        cfg.use_gpu = use_gpu;

        printf("\\n🔧 Extracting with %s...\\n", use_gpu ?  "GPU" : "CPU");

        Watermark *extracted;
        if (use_gpu) {
            extracted = extract_watermark_cuda(img, &cfg);
        } else {
            extracted = extract_watermark_cpu(img, &cfg);
        }

        printf("   Extracted:  %dx%d\\n", extracted->width, extracted->height);

        Image *wm_img = (Image*)malloc(sizeof(Image));
        wm_img->width = extracted->width;
        wm_img->height = extracted->height;
        wm_img->channels = 3;
        wm_img->data = (unsigned char*)malloc(extracted->width * extracted->height * 3);
        for (int i = 0; i < extracted->width * extracted->height; i++) {
            wm_img->data[i*3] = wm_img->data[i*3+1] = wm_img->data[i*3+2] = extracted->data[i];
        }
        save_image("results/extracted_watermark.png", wm_img);

        printf("\\n✅ Success!\\n");

        free_image(wm_img);
        free_watermark(extracted);
        free_image(img);

    } else if (strcmp(mode, "check") == 0) {
        if (argc < 5) {
            fprintf(stderr, "Usage: %s check <watermarked_image> <original_logo> <gpu|cpu>\\n", argv[0]);
            return 1;
        }

        printf("\\n╔════════════════════════════════════╗\\n");
        printf("║   CHECK WATERMARK                 ║\\n");
        printf("╚════════════════════════════════════╝\\n\\n");

        Image *img = load_image(argv[2]);
        if (!img) return 1;

        bool use_gpu = (strcmp(argv[4], "gpu") == 0);
        Config cfg;
        cfg.alpha = 1;
        cfg.use_gpu = use_gpu;

        printf("\\n🔧 Extracting watermark with %s...\\n", use_gpu ?  "GPU" : "CPU");

        Watermark *extracted;
        if (use_gpu) {
            extracted = extract_watermark_cuda(img, &cfg);
        } else {
            extracted = extract_watermark_cpu(img, &cfg);
        }

        printf("   Extracted size: %dx%d\\n", extracted->width, extracted->height);

        printf("\\n🔧 Loading original logo...\\n");
        Watermark *original_logo = load_logo(argv[3], extracted->width, extracted->height);

        if (!original_logo) {
            fprintf(stderr, "   Error loading logo\\n");
            free_image(img);
            free_watermark(extracted);
            return 1;
        }

        printf("   Original size: %dx%d\\n", original_logo->width, original_logo->height);

        printf("\\n🔧 Comparing... \\n");
        double similarity = calculate_similarity(original_logo, extracted);

        printf("\\n📊 RESULTS:\\n");
        printf("   Similarity:  %.2f%%%%\\n", similarity);

        if (similarity > 90.0) {
            printf("   ✅ VERIFIED - Watermark Present! \\n");
        } else if (similarity > 70.0) {
            printf("   ⚠️  PARTIAL - Possible Match\\n");
        } else {
            printf("   ❌ NOT FOUND - No Watermark\\n");
        }

        free_image(img);
        free_watermark(extracted);
        free_watermark(original_logo);
    }

    clock_t end = clock();
    double time_spent = (double)(end - start) / CLOCKS_PER_SEC;
    printf("\\n⏱️  Time:  %.3f sec\\n\\n", time_spent);

    return 0;
}
""")

    with open('Makefile', 'w') as f:
        f.write("""CC = gcc
NVCC = nvcc
CFLAGS = -O3 -Wall
NVCCFLAGS = -O3 -arch=sm_75
LDFLAGS = -lm

all: dirs bin/watermark

dirs:
\tmkdir -p build bin results uploads

build/image_utils.o: src/image_utils.c
\t$(CC) $(CFLAGS) -Iinclude -c -o $@ $<

build/watermark_cpu.o: src/watermark_cpu.c
\t$(CC) $(CFLAGS) -Iinclude -c -o $@ $<

build/watermark_cuda.o: src/watermark_cuda.cu
\t$(NVCC) $(NVCCFLAGS) -Iinclude -c -o $@ $<

build/main.o: src/main.c
\t$(CC) $(CFLAGS) -Iinclude -c -o $@ $<

bin/watermark: build/main.o build/image_utils.o build/watermark_cuda.o build/watermark_cpu.o
\t$(NVCC) $(NVCCFLAGS) -o $@ $^ $(LDFLAGS)

clean:
\trm -rf build bin results

.PHONY: all dirs clean
""")

    print("✅ Source files created!\n")

# ============================================================================
# COMPILE
# ============================================================================

def compile_code():
    """Compile the code"""
    print("="*60)
    print("🔨 COMPILING")
    print("="*60)

    result = subprocess.run("make clean && make all", shell=True,
                          capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("❌ Failed:")
        print(result.stderr)
        return False

    print("✅ Compiled!\n")
    return True

# ============================================================================
# UPLOAD
# ============================================================================

def upload_image(prompt_text):
    """Upload an image file"""
    print(f"\n📤 {prompt_text}")
    uploaded = files.upload()

    if not uploaded:
        return None

    filename = list(uploaded.keys())[0]
    filepath = os.path.join('uploads', filename)

    with open(filepath, 'wb') as f:
        f.write(uploaded[filename])

    print(f"✅ Uploaded: {filename}")
    return filepath

# ============================================================================
# MENU - SHORTER BARS
# ============================================================================

def display_menu():
    """Display menu"""
    print("\n" + "="*50)
    print(" "*10 + "🎨 WATERMARKING SYSTEM")
    print("="*50)
    print("\n📋 OPTIONS:\n")
    print("  1️⃣  Image Similarity")
    print("  2️⃣  Embed Watermark")
    print("  3️⃣  Verify Watermark")
    print("  4️⃣  Extract Watermark")
    print("  5️⃣  Exit")
    print("\n" + "="*50)

from IPython.display import display, HTML
import base64

def run_interactive_menu():
    """Run menu"""

    while True:
        display_menu()
        choice = input("\n👉 Choice (1-5): ").strip()

        if choice == '1':
            print("\n" + "="*50)
            print("1️⃣  IMAGE SIMILARITY")
            print("="*50)

            img1_path = upload_image("Upload IMAGE 1:")
            if not img1_path:
                continue

            img2_path = upload_image("Upload IMAGE 2:")
            if not img2_path:
                continue

            print("\n🔄 Processing...")
            cmd = ["bin/watermark", "similarity", img1_path, img2_path]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            img1 = Image. open(img1_path)
            img2 = Image.open(img2_path)
            ax1.imshow(img1)
            ax1.set_title('Image 1', fontsize=12, fontweight='bold')
            ax1.axis('off')
            ax2.imshow(img2)
            ax2.set_title('Image 2', fontsize=12, fontweight='bold')
            ax2.axis('off')
            plt.tight_layout()
            plt.show()

        elif choice == '2':
            print("\n" + "="*50)
            print("2️⃣  EMBED WATERMARK")
            print("="*50)

            img_path = upload_image("Upload IMAGE:")
            if not img_path:
                continue

            logo_path = upload_image("Upload LOGO:")
            if not logo_path:
                continue

            mode = input("\n⚙️  Mode [gpu/cpu]:  ").strip().lower()
            if mode not in ['gpu', 'cpu']:
                mode = 'gpu'

            print(f"\n🔄 Embedding ({mode.upper()})...")
            cmd = ["bin/watermark", "embed", img_path, logo_path, mode]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            # Check for the file that C program creates
            output_file = 'results/watermarked_image.png'

            # Debug: Show what files exist
            if os.path.exists('results'):
                print(f"\n📁 Files in results/: {os.listdir('results')}")

            if os.path.exists(output_file):
                print(f"✅ Done! ({os. path.getsize(output_file)} bytes)")

                try:
                    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
                    ax1.imshow(Image.open(img_path))
                    ax1.set_title('Original', fontsize=11, fontweight='bold')
                    ax1.axis('off')

                    ax2.imshow(Image.open(logo_path))
                    ax2.set_title('Logo', fontsize=11, fontweight='bold')
                    ax2.axis('off')

                    ax3.imshow(Image.open(output_file))
                    ax3.set_title('Watermarked', fontsize=11, fontweight='bold')
                    ax3.axis('off')

                    plt. tight_layout()
                    plt.show()
                except Exception as e:
                    print(f"⚠️  Display error:  {e}")

                # IMMEDIATE DOWNLOAD - Create clickable link
                print("\n💾 Preparing download...")
                try:
                    with open(output_file, 'rb') as f:
                        data = f.read()
                        b64 = base64.b64encode(data).decode()
                        href = f'<a href="data:image/png;base64,{b64}" download="watermarked_image.png" style="font-size:16px; padding:10px; background:#4CAF50; color: white; text-decoration:none; border-radius:5px;">📥 DOWNLOAD WATERMARKED IMAGE</a>'
                        display(HTML(href))
                    print("✅ Click the green button above to download ⬆️")
                except Exception as e:
                    print(f"❌ Download error: {e}")
            else:
                print(f"❌ Failed!  Output file not found: {output_file}")
                if os.path.exists('results'):
                    print(f"Available files:  {os.listdir('results')}")

        elif choice == '3':
            print("\n" + "="*50)
            print("3️⃣  VERIFY WATERMARK")
            print("="*50)

            img_path = upload_image("Upload WATERMARKED IMAGE:")
            if not img_path:
                continue

            logo_path = upload_image("Upload ORIGINAL LOGO:")
            if not logo_path:
                continue

            mode = input("\n⚙️  Mode [gpu/cpu]: ").strip().lower()
            if mode not in ['gpu', 'cpu']:
                mode = 'gpu'

            print(f"\n🔄 Verifying ({mode.upper()})...")
            cmd = ["bin/watermark", "check", img_path, logo_path, mode]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            ax1.imshow(Image.open(img_path))
            ax1.set_title('Watermarked', fontsize=11, fontweight='bold')
            ax1.axis('off')
            ax2.imshow(Image. open(logo_path))
            ax2.set_title('Original Logo', fontsize=11, fontweight='bold')
            ax2.axis('off')
            plt.tight_layout()
            plt.show()

        elif choice == '4':
            print("\n" + "="*50)
            print("4️⃣  EXTRACT WATERMARK")
            print("="*50)

            img_path = upload_image("Upload WATERMARKED IMAGE:")
            if not img_path:
                continue

            mode = input("\n⚙️  Mode [gpu/cpu]: ").strip().lower()
            if mode not in ['gpu', 'cpu']:
                mode = 'gpu'

            print(f"\n🔄 Extracting ({mode.upper()})...")
            cmd = ["bin/watermark", "extract", img_path, mode]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            output_file = 'results/extracted_watermark.png'

            # Debug: Show what files exist
            if os.path.exists('results'):
                print(f"\n📁 Files in results/:  {os.listdir('results')}")

            if os. path.exists(output_file):
                try:
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
                    ax1.imshow(Image.open(img_path))
                    ax1.set_title('Watermarked', fontsize=11, fontweight='bold')
                    ax1.axis('off')

                    ax2.imshow(Image.open(output_file), cmap='gray')
                    ax2.set_title('Extracted', fontsize=11, fontweight='bold')
                    ax2.axis('off')

                    plt.tight_layout()
                    plt.show()
                except Exception as e:
                    print(f"Display error: {e}")

                # IMMEDIATE DOWNLOAD - Create clickable link
                print("\n💾 Preparing download...")
                try:
                    with open(output_file, 'rb') as f:
                        data = f.read()
                        b64 = base64.b64encode(data).decode()
                        href = f'<a href="data:image/png;base64,{b64}" download="extracted_watermark.png" style="font-size:16px; padding:10px; background:#2196F3; color:white; text-decoration:none; border-radius:5px;">📥 DOWNLOAD EXTRACTED WATERMARK</a>'
                        display(HTML(href))
                    print("✅ Click the blue button above to download ⬆️")
                except Exception as e:
                    print(f"❌ Download error: {e}")
            else:
                print(f"❌ Failed!  Output file not found: {output_file}")
                if os.path.exists('results'):
                    print(f"Available files: {os.listdir('results')}")

        elif choice == '5':
            print("\n👋 Goodbye!\n")
            break

        else:
            print("\n❌ Invalid!  Enter 1-5.")

        input("\n⏸️  [Press Enter]")

# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main function"""
    print("\n" + "="*60)
    print(" "*10 + "IMAGE WATERMARKING SYSTEM")
    print(" "*8 + "MPI + CUDA for Google Colab")
    print("="*60 + "\n")

    setup_environment()
    create_source_files()

    if compile_code():
        print("\n✅ Ready!")
        run_interactive_menu()
    else:
        print("\n❌ Compilation failed!")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Interactive Image Watermarking System with MPI + CUDA
COMPLETE MPI+CUDA VERSION - Ready for Google Colab
"""

import os
import subprocess
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display, HTML
import base64
import io

# ============================================================================
# STEP 1: SETUP ENVIRONMENT
# ============================================================================

def setup_environment():
    """Install all dependencies"""
    print("="*60)
    print("🔧 SETTING UP ENVIRONMENT")
    print("="*60)

    print("\n📊 GPU Information:")
    os.system("nvidia-smi --query-gpu=name,memory.total --format=csv")

    print("\n📦 Installing dependencies...")
    os.system("apt-get update -qq && apt-get install -y -qq build-essential libopenmpi-dev openmpi-bin libpng-dev libjpeg-dev wget > /dev/null 2>&1")
    os.system("pip install -q numpy matplotlib pillow pandas")

    for d in ['src', 'include', 'build', 'bin', 'data', 'results', 'uploads']:
        os.makedirs(d, exist_ok=True)

    print("\n📥 Downloading headers...")
    os.system("wget -q -O include/stb_image.h https://raw.githubusercontent.com/nothings/stb/master/stb_image.h")
    os.system("wget -q -O include/stb_image_write.h https://raw.githubusercontent.com/nothings/stb/master/stb_image_write.h")

    print("✅ Setup complete!\n")

# ============================================================================
# STEP 2: CREATE SOURCE FILES WITH MPI + CUDA
# ============================================================================

def create_source_files():
    """Generate all source code files with MPI support"""
    print("="*60)
    print("💾 CREATING SOURCE FILES (MPI + CUDA)")
    print("="*60)

    with open('include/watermark.h', 'w') as f:
        f.write("""#ifndef WATERMARK_H
#define WATERMARK_H

#include <stdint.h>
#include <stdbool.h>

typedef struct {
    int width;
    int height;
    int channels;
    unsigned char *data;
} Image;

typedef struct {
    int width;
    int height;
    unsigned char *data;
} Watermark;

typedef struct {
    int alpha;
    bool use_gpu;
    int mpi_rank;
    int mpi_size;
} Config;

#ifdef __cplusplus
extern "C" {
#endif

Image* load_image(const char *filename);
void save_image(const char *filename, Image *img);
void free_image(Image *img);
Watermark* load_logo(const char *filename, int target_width, int target_height);
void free_watermark(Watermark *wm);
void embed_watermark_cuda(Image *img, Watermark *wm, Config *config, int start_row, int end_row);
Watermark* extract_watermark_cuda(Image *img, Config *config);
void embed_watermark_cpu(Image *img, Watermark *wm, Config *config, int start_row, int end_row);
Watermark* extract_watermark_cpu(Image *img, Config *config);
double calculate_similarity(Watermark *wm1, Watermark *wm2);
double calculate_image_similarity(Image *img1, Image *img2);
double calculate_psnr(Image *img1, Image *img2);

#ifdef __cplusplus
}
#endif

#endif
""")

    with open('src/image_utils.c', 'w') as f:
        f.write("""#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#define STB_IMAGE_IMPLEMENTATION
#define STB_IMAGE_WRITE_IMPLEMENTATION
#include "../include/stb_image.h"
#include "../include/stb_image_write.h"
#include "../include/watermark.h"

Image* load_image(const char *filename) {
    Image *img = (Image*)malloc(sizeof(Image));
    img->data = stbi_load(filename, &img->width, &img->height, &img->channels, 3);
    if (!img->data) {
        fprintf(stderr, "Error loading:  %s\\n", filename);
        free(img);
        return NULL;
    }
    img->channels = 3;
    printf("✓ Loaded:  %dx%d\\n", img->width, img->height);
    return img;
}

void save_image(const char *filename, Image *img) {
    stbi_write_png(filename, img->width, img->height, img->channels,
                   img->data, img->width * img->channels);
    printf("✓ Saved: %s\\n", filename);
}

void free_image(Image *img) {
    if (img) {
        if (img->data) stbi_image_free(img->data);
        free(img);
    }
}

Watermark* load_logo(const char *filename, int target_width, int target_height) {
    int orig_width, orig_height, orig_channels;
    unsigned char *orig_data = stbi_load(filename, &orig_width, &orig_height, &orig_channels, 1);

    if (!orig_data) {
        fprintf(stderr, "Error loading logo: %s\\n", filename);
        return NULL;
    }

    Watermark *wm = (Watermark*)malloc(sizeof(Watermark));
    wm->width = target_width;
    wm->height = target_height;
    wm->data = (unsigned char*)malloc(target_width * target_height);

    for (int y = 0; y < target_height; y++) {
        for (int x = 0; x < target_width; x++) {
            int src_x = (x * orig_width) / target_width;
            int src_y = (y * orig_height) / target_height;
            unsigned char pixel = orig_data[src_y * orig_width + src_x];
            wm->data[y * target_width + x] = (pixel > 128) ? 255 : 0;
        }
    }
    stbi_image_free(orig_data);
    printf("✓ Logo loaded: %dx%d\\n", target_width, target_height);
    return wm;
}

void free_watermark(Watermark *wm) {
    if (wm) {
        if (wm->data) free(wm->data);
        free(wm);
    }
}

double calculate_similarity(Watermark *wm1, Watermark *wm2) {
    if (wm1->width != wm2->width || wm1->height != wm2->height) return 0.0;
    int matches = 0;
    int total = wm1->width * wm1->height;
    for (int i = 0; i < total; i++) {
        if (wm1->data[i] == wm2->data[i]) matches++;
    }
    return (double)matches / total * 100.0;
}

double calculate_image_similarity(Image *img1, Image *img2) {
    if (img1->width != img2->width || img1->height != img2->height) return 0.0;
    int size = img1->width * img1->height * img1->channels;
    long long sum_sq_diff = 0;

    for (int i = 0; i < size; i++) {
        int diff = (int)img1->data[i] - (int)img2->data[i];
        sum_sq_diff += diff * diff;
    }

    double mse = (double)sum_sq_diff / size;
    double similarity = 100.0 / (1.0 + sqrt(mse));
    return similarity;
}

double calculate_psnr(Image *img1, Image *img2) {
    if (img1->width != img2->width || img1->height != img2->height) return 0.0;
    int size = img1->width * img1->height * img1->channels;
    double mse = 0.0;
    for (int i = 0; i < size; i++) {
        double diff = (double)img1->data[i] - (double)img2->data[i];
        mse += diff * diff;
    }
    mse /= size;
    if (mse == 0) return INFINITY;
    return 10.0 * log10((255.0 * 255.0) / mse);
}
""")

    # MPI + CUDA - LSB EMBEDDING WITH ROW DISTRIBUTION
    with open('src/watermark_cuda.cu', 'w') as f:
        f.write("""#include <cuda_runtime.h>
#include <stdio.h>
#include "../include/watermark.h"

#define BLOCK_SIZE 16
#define CUDA_CHECK(call) { \\
    cudaError_t err = call; \\
    if (err != cudaSuccess) { \\
        fprintf(stderr, "CUDA error: %s\\n", cudaGetErrorString(err)); \\
        exit(EXIT_FAILURE); \\
    } \\
}

__global__ void embed_lsb_kernel(unsigned char *img, unsigned char *wm,
                                  int img_width, int img_height,
                                  int wm_width, int wm_height,
                                  int start_row, int end_row) {
    int x = blockIdx.x * blockDim.x + threadIdx. x;
    int y = blockIdx.y * blockDim. y + threadIdx.y + start_row;

    if (x >= img_width || y >= end_row || y >= img_height) return;

    int wm_x = (x * wm_width) / img_width;
    int wm_y = (y * wm_height) / img_height;

    if (wm_x >= wm_width || wm_y >= wm_height) return;

    unsigned char wm_bit = wm[wm_y * wm_width + wm_x];
    int img_idx = (y * img_width + x) * 3;

    if (wm_bit > 128) {
        img[img_idx] |= 1;
        img[img_idx + 1] |= 1;
        img[img_idx + 2] |= 1;
    } else {
        img[img_idx] &= 0xFE;
        img[img_idx + 1] &= 0xFE;
        img[img_idx + 2] &= 0xFE;
    }
}

__global__ void extract_lsb_kernel(unsigned char *img, unsigned char *wm,
                                     int img_width, int img_height,
                                     int wm_width, int wm_height) {
    int wm_x = blockIdx.x * blockDim. x + threadIdx.x;
    int wm_y = blockIdx.y * blockDim. y + threadIdx.y;

    if (wm_x >= wm_width || wm_y >= wm_height) return;

    int img_x = (wm_x * img_width) / wm_width;
    int img_y = (wm_y * img_height) / wm_height;

    int img_idx = (img_y * img_width + img_x) * 3;

    int bit_count = 0;
    bit_count += (img[img_idx] & 1);
    bit_count += (img[img_idx + 1] & 1);
    bit_count += (img[img_idx + 2] & 1);

    wm[wm_y * wm_width + wm_x] = (bit_count >= 2) ? 255 : 0;
}

void embed_watermark_cuda(Image *img, Watermark *wm, Config *cfg, int start_row, int end_row) {
    int rows_to_process = end_row - start_row;
    int img_size = img->width * img->height * 3;
    int wm_size = wm->width * wm->height;

    unsigned char *d_img, *d_wm;

    CUDA_CHECK(cudaMalloc(&d_img, img_size));
    CUDA_CHECK(cudaMalloc(&d_wm, wm_size));

    CUDA_CHECK(cudaMemcpy(d_img, img->data, img_size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_wm, wm->data, wm_size, cudaMemcpyHostToDevice));

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((img->width + BLOCK_SIZE - 1) / BLOCK_SIZE,
              (rows_to_process + BLOCK_SIZE - 1) / BLOCK_SIZE);

    embed_lsb_kernel<<<grid, block>>>(d_img, d_wm, img->width, img->height,
                                       wm->width, wm->height, start_row, end_row);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(img->data, d_img, img_size, cudaMemcpyDeviceToHost));

    cudaFree(d_img);
    cudaFree(d_wm);
}

Watermark* extract_watermark_cuda(Image *img, Config *cfg) {
    int wm_width = img->width / 4;
    int wm_height = img->height / 4;

    Watermark *wm = (Watermark*)malloc(sizeof(Watermark));
    wm->width = wm_width;
    wm->height = wm_height;
    wm->data = (unsigned char*)malloc(wm_width * wm_height);

    int img_size = img->width * img->height * 3;
    int wm_size = wm_width * wm_height;

    unsigned char *d_img, *d_wm;

    CUDA_CHECK(cudaMalloc(&d_img, img_size));
    CUDA_CHECK(cudaMalloc(&d_wm, wm_size));

    CUDA_CHECK(cudaMemcpy(d_img, img->data, img_size, cudaMemcpyHostToDevice));

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((wm_width + BLOCK_SIZE - 1) / BLOCK_SIZE,
              (wm_height + BLOCK_SIZE - 1) / BLOCK_SIZE);

    extract_lsb_kernel<<<grid, block>>>(d_img, d_wm, img->width, img->height,
                                         wm_width, wm_height);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(wm->data, d_wm, wm_size, cudaMemcpyDeviceToHost));

    cudaFree(d_img);
    cudaFree(d_wm);

    return wm;
}
""")

    # MPI + CPU VERSION
    with open('src/watermark_cpu.c', 'w') as f:
        f.write("""#include <stdio. h>
#include <stdlib. h>
#include "../include/watermark.h"

void embed_watermark_cpu(Image *img, Watermark *wm, Config *cfg, int start_row, int end_row) {
    for (int y = start_row; y < end_row && y < img->height; y++) {
        for (int x = 0; x < img->width; x++) {
            int wm_x = (x * wm->width) / img->width;
            int wm_y = (y * wm->height) / img->height;

            unsigned char wm_bit = wm->data[wm_y * wm->width + wm_x];
            int img_idx = (y * img->width + x) * 3;

            if (wm_bit > 128) {
                img->data[img_idx] |= 1;
                img->data[img_idx + 1] |= 1;
                img->data[img_idx + 2] |= 1;
            } else {
                img->data[img_idx] &= 0xFE;
                img->data[img_idx + 1] &= 0xFE;
                img->data[img_idx + 2] &= 0xFE;
            }
        }
    }
}

Watermark* extract_watermark_cpu(Image *img, Config *cfg) {
    int wm_width = img->width / 4;
    int wm_height = img->height / 4;

    Watermark *wm = (Watermark*)malloc(sizeof(Watermark));
    wm->width = wm_width;
    wm->height = wm_height;
    wm->data = (unsigned char*)malloc(wm_width * wm_height);

    for (int wm_y = 0; wm_y < wm_height; wm_y++) {
        for (int wm_x = 0; wm_x < wm_width; wm_x++) {
            int img_x = (wm_x * img->width) / wm_width;
            int img_y = (wm_y * img->height) / wm_height;

            int img_idx = (img_y * img->width + img_x) * 3;

            int bit_count = 0;
            bit_count += (img->data[img_idx] & 1);
            bit_count += (img->data[img_idx + 1] & 1);
            bit_count += (img->data[img_idx + 2] & 1);

            wm->data[wm_y * wm_width + wm_x] = (bit_count >= 2) ? 255 : 0;
        }
    }

    return wm;
}
""")

    # MPI MAIN PROGRAM
    with open('src/main.c', 'w') as f:
        f.write("""#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include "../include/watermark.h"

int main(int argc, char **argv) {
    int rank, size;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (argc < 2) {
        if (rank == 0) fprintf(stderr, "Usage: %s <mode> [args]\\n", argv[0]);
        MPI_Finalize();
        return 1;
    }

    char *mode = argv[1];
    double start_time = MPI_Wtime();

    if (strcmp(mode, "embed") == 0 && argc >= 5) {
        if (rank == 0) {
            printf("\\n╔════════════════════════════════════╗\\n");
            printf("║   EMBED WATERMARK (MPI+CUDA)      ║\\n");
            printf("╚════════════════════════════════════╝\\n");
            printf("   MPI Processes: %d\\n\\n", size);
        }

        Image *img = NULL;
        Watermark *logo = NULL;
        int img_width, img_height, wm_size;

        if (rank == 0) {
            img = load_image(argv[2]);
            if (!img) {
                MPI_Abort(MPI_COMM_WORLD, 1);
            }

            wm_size = img->width / 4;
            logo = load_logo(argv[3], wm_size, wm_size);
            if (!logo) {
                free_image(img);
                MPI_Abort(MPI_COMM_WORLD, 1);
            }

            img_width = img->width;
            img_height = img->height;
            printf("   Watermark size: %dx%d\\n", wm_size, wm_size);
        }

        MPI_Bcast(&img_width, 1, MPI_INT, 0, MPI_COMM_WORLD);
        MPI_Bcast(&img_height, 1, MPI_INT, 0, MPI_COMM_WORLD);
        MPI_Bcast(&wm_size, 1, MPI_INT, 0, MPI_COMM_WORLD);

        if (rank != 0) {
            img = (Image*)malloc(sizeof(Image));
            img->width = img_width;
            img->height = img_height;
            img->channels = 3;
            img->data = (unsigned char*)malloc(img_width * img_height * 3);

            logo = (Watermark*)malloc(sizeof(Watermark));
            logo->width = wm_size;
            logo->height = wm_size;
            logo->data = (unsigned char*)malloc(wm_size * wm_size);
        }

        MPI_Bcast(img->data, img_width * img_height * 3, MPI_UNSIGNED_CHAR, 0, MPI_COMM_WORLD);
        MPI_Bcast(logo->data, wm_size * wm_size, MPI_UNSIGNED_CHAR, 0, MPI_COMM_WORLD);

        int rows_per_proc = img_height / size;
        int start_row = rank * rows_per_proc;
        int end_row = (rank == size - 1) ? img_height : (rank + 1) * rows_per_proc;

        if (rank == 0) printf("\\n🔧 Processing with MPI+GPU (Rank %d:  rows %d-%d)...\\n", rank, start_row, end_row);

        bool use_gpu = (strcmp(argv[4], "gpu") == 0);
        Config cfg = {1, use_gpu, rank, size};

        if (use_gpu) {
            embed_watermark_cuda(img, logo, &cfg, start_row, end_row);
        } else {
            embed_watermark_cpu(img, logo, &cfg, start_row, end_row);
        }

        if (rank == 0) {
            for (int r = 1; r < size; r++) {
                int r_start = r * rows_per_proc;
                int r_end = (r == size - 1) ? img_height : (r + 1) * rows_per_proc;
                int r_rows = r_end - r_start;
                MPI_Recv(img->data + r_start * img_width * 3, r_rows * img_width * 3,
                         MPI_UNSIGNED_CHAR, r, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            }

            save_image("results/watermarked_image.png", img);
            printf("\\n✅ Success! \\n");
        } else {
            int my_rows = end_row - start_row;
            MPI_Send(img->data + start_row * img_width * 3, my_rows * img_width * 3,
                     MPI_UNSIGNED_CHAR, 0, 0, MPI_COMM_WORLD);
        }

        free_image(img);
        free_watermark(logo);

    } else if (strcmp(mode, "extract") == 0 && argc >= 4) {
        if (rank == 0) {
            printf("\\n╔════════════════════════════════════╗\\n");
            printf("║   EXTRACT WATERMARK               ║\\n");
            printf("╚════════════════════════════════════╝\\n\\n");

            Image *img = load_image(argv[2]);
            if (!img) {
                MPI_Abort(MPI_COMM_WORLD, 1);
            }

            bool use_gpu = (strcmp(argv[3], "gpu") == 0);
            Config cfg = {1, use_gpu, 0, 1};

            printf("\\n🔧 Extracting with %s... \\n", use_gpu ? "GPU" : "CPU");

            Watermark *extracted;
            if (use_gpu) {
                extracted = extract_watermark_cuda(img, &cfg);
            } else {
                extracted = extract_watermark_cpu(img, &cfg);
            }

            printf("   Extracted:  %dx%d\\n", extracted->width, extracted->height);

            Image *wm_img = (Image*)malloc(sizeof(Image));
            wm_img->width = extracted->width;
            wm_img->height = extracted->height;
            wm_img->channels = 3;
            wm_img->data = (unsigned char*)malloc(extracted->width * extracted->height * 3);
            for (int i = 0; i < extracted->width * extracted->height; i++) {
                wm_img->data[i*3] = wm_img->data[i*3+1] = wm_img->data[i*3+2] = extracted->data[i];
            }
            save_image("results/extracted_watermark.png", wm_img);

            printf("\\n✅ Success!\\n");

            free_image(wm_img);
            free_watermark(extracted);
            free_image(img);
        }

    } else if (strcmp(mode, "check") == 0 && argc >= 5) {
        if (rank == 0) {
            printf("\\n╔════════════════════════════════════╗\\n");
            printf("║   CHECK WATERMARK                 ║\\n");
            printf("╚════════════════════════════════════╝\\n\\n");

            Image *img = load_image(argv[2]);
            if (!img) {
                MPI_Abort(MPI_COMM_WORLD, 1);
            }

            bool use_gpu = (strcmp(argv[4], "gpu") == 0);
            Config cfg = {1, use_gpu, 0, 1};

            printf("\\n🔧 Extracting watermark with %s...\\n", use_gpu ? "GPU" : "CPU");

            Watermark *extracted;
            if (use_gpu) {
                extracted = extract_watermark_cuda(img, &cfg);
            } else {
                extracted = extract_watermark_cpu(img, &cfg);
            }

            printf("   Extracted size: %dx%d\\n", extracted->width, extracted->height);

            printf("\\n🔧 Loading original logo...\\n");
            Watermark *original_logo = load_logo(argv[3], extracted->width, extracted->height);

            if (! original_logo) {
                fprintf(stderr, "   Error loading logo\\n");
                free_image(img);
                free_watermark(extracted);
                MPI_Abort(MPI_COMM_WORLD, 1);
            }

            printf("   Original size: %dx%d\\n", original_logo->width, original_logo->height);

            printf("\\n🔧 Comparing...\\n");
            double similarity = calculate_similarity(original_logo, extracted);

            printf("\\n📊 RESULTS:\\n");
            printf("   Similarity: %.2f%%%%\\n", similarity);

            if (similarity > 90.0) {
                printf("   ✅ VERIFIED - Watermark Present! \\n");
            } else if (similarity > 70.0) {
                printf("   ⚠️  PARTIAL - Possible Match\\n");
            } else {
                printf("   ❌ NOT FOUND - No Watermark\\n");
            }

            free_image(img);
            free_watermark(extracted);
            free_watermark(original_logo);
        }

    } else if (strcmp(mode, "similarity") == 0 && argc >= 4) {
        if (rank == 0) {
            printf("\\n╔════════════════════════════════════╗\\n");
            printf("║   IMAGE SIMILARITY CHECK          ║\\n");
            printf("╚════════════════════════════════════╝\\n\\n");

            Image *img1 = load_image(argv[2]);
            Image *img2 = load_image(argv[3]);

            if (!img1 || !img2) {
                fprintf(stderr, "Error loading images\\n");
                MPI_Abort(MPI_COMM_WORLD, 1);
            }

            double similarity = calculate_image_similarity(img1, img2);
            double psnr = calculate_psnr(img1, img2);

            printf("\\n📊 RESULTS:\\n");
            printf("   Similarity: %.2f%%%%\\n", similarity);
            printf("   PSNR: %.2f dB\\n", psnr);

            if (similarity > 95.0) {
                printf("   ✅ VERY SIMILAR\\n");
            } else if (similarity > 80.0) {
                printf("   ✅ SIMILAR\\n");
            } else if (similarity > 60.0) {
                printf("   ⚠️  SOMEWHAT SIMILAR\\n");
            } else {
                printf("   ❌ DIFFERENT\\n");
            }

            free_image(img1);
            free_image(img2);
        }
    }

    if (rank == 0) {
        double end_time = MPI_Wtime();
        printf("\\n⏱️  Time:  %.3f sec (MPI:  %d processes)\\n\\n", end_time - start_time, size);
    }

    MPI_Finalize();
    return 0;
}
""")

    # UPDATED MAKEFILE WITH MPI
    with open('Makefile', 'w') as f:
        f.write("""CC = gcc
MPICC = mpicc
NVCC = nvcc
CFLAGS = -O3 -Wall
MPIFLAGS = -O3 -Wall
NVCCFLAGS = -O3 -arch=sm_75
LDFLAGS = -lm

all: dirs bin/watermark

dirs:
\tmkdir -p build bin results uploads

build/image_utils.o: src/image_utils.c
\t$(CC) $(CFLAGS) -Iinclude -c -o $@ $<

build/watermark_cpu.o: src/watermark_cpu.c
\t$(CC) $(CFLAGS) -Iinclude -c -o $@ $<

build/watermark_cuda.o: src/watermark_cuda.cu
\t$(NVCC) $(NVCCFLAGS) -Iinclude -c -o $@ $<

build/main.o: src/main.c
\t$(MPICC) $(MPIFLAGS) -Iinclude -c -o $@ $<

bin/watermark: build/main.o build/image_utils.o build/watermark_cuda.o build/watermark_cpu.o
\t$(MPICC) -o $@ $^ $(LDFLAGS) -L/usr/local/cuda/lib64 -lcudart

clean:
\trm -rf build bin results

. PHONY: all dirs clean
""")

    print("✅ MPI + CUDA source files created!\n")

# ============================================================================
# COMPILE
# ============================================================================

def compile_code():
    """Compile the code"""
    print("="*60)
    print("🔨 COMPILING MPI + CUDA")
    print("="*60)

    result = subprocess.run("make clean && make all", shell=True,
                          capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("❌ Failed:")
        print(result.stderr)
        return False

    print("✅ Compiled!\n")
    return True

# ============================================================================
# UPLOAD
# ============================================================================

def upload_image(prompt_text):
    """Upload an image file"""
    print(f"\n📤 {prompt_text}")
    uploaded = files.upload()

    if not uploaded:
        return None

    filename = list(uploaded.keys())[0]
    filepath = os.path.join('uploads', filename)

    with open(filepath, 'wb') as f:
        f.write(uploaded[filename])

    print(f"✅ Uploaded: {filename}")
    return filepath

# ============================================================================
# MENU
# ============================================================================

def display_menu():
    """Display menu"""
    print("\n" + "="*50)
    print(" "*8 + "🎨 MPI+CUDA WATERMARKING")
    print("="*50)
    print("\n📋 OPTIONS:\n")
    print("  1️⃣  Image Similarity")
    print("  2️⃣  Embed Watermark (MPI+CUDA)")
    print("  3️⃣  Verify Watermark")
    print("  4️⃣  Extract Watermark")
    print("  5️⃣  Exit")
    print("\n" + "="*50)

def run_interactive_menu():
    """Run menu with MPI support"""

    while True:
        display_menu()
        choice = input("\n👉 Choice (1-5): ").strip()

        if choice == '1':
            print("\n" + "="*50)
            print("1️⃣  IMAGE SIMILARITY")
            print("="*50)

            img1_path = upload_image("Upload IMAGE 1:")
            if not img1_path:
                continue

            img2_path = upload_image("Upload IMAGE 2:")
            if not img2_path:
                continue

            print("\n🔄 Processing with MPI...")
            cmd = ["mpirun", "--allow-run-as-root", "-np", "2", "bin/watermark", "similarity", img1_path, img2_path]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            img1 = Image.open(img1_path)
            img2 = Image. open(img2_path)
            ax1.imshow(img1)
            ax1.set_title('Image 1', fontsize=12, fontweight='bold')
            ax1.axis('off')
            ax2.imshow(img2)
            ax2.set_title('Image 2', fontsize=12, fontweight='bold')
            ax2.axis('off')
            plt.tight_layout()
            plt.show()

        elif choice == '2':
            print("\n" + "="*50)
            print("2️⃣  EMBED WATERMARK (MPI+CUDA)")
            print("="*50)

            img_path = upload_image("Upload IMAGE:")
            if not img_path:
                continue

            logo_path = upload_image("Upload LOGO:")
            if not logo_path:
                continue

            mode = input("\n⚙️  Mode [gpu/cpu]: ").strip().lower()
            if mode not in ['gpu', 'cpu']:
                mode = 'gpu'

            num_procs = input("⚙️  MPI Processes [2]: ").strip()
            if not num_procs or not num_procs.isdigit():
                num_procs = "2"

            print(f"\n🔄 Embedding with MPI ({num_procs} processes) + {mode. upper()}...")
            cmd = ["mpirun", "--allow-run-as-root", "-np", num_procs, "bin/watermark", "embed", img_path, logo_path, mode]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            output_file = 'results/watermarked_image.png'

            if os.path.exists(output_file):
                print(f"✅ Done! ({os.path.getsize(output_file)} bytes)")

                try:
                    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
                    ax1.imshow(Image.open(img_path))
                    ax1.set_title('Original', fontsize=11, fontweight='bold')
                    ax1.axis('off')

                    ax2.imshow(Image.open(logo_path))
                    ax2.set_title('Logo', fontsize=11, fontweight='bold')
                    ax2.axis('off')

                    ax3.imshow(Image. open(output_file))
                    ax3.set_title('Watermarked', fontsize=11, fontweight='bold')
                    ax3.axis('off')

                    plt.tight_layout()
                    plt.show()
                except Exception as e:
                    print(f"⚠️  Display error: {e}")

                print("\n💾 Preparing download...")
                try:
                    with open(output_file, 'rb') as f:
                        data = f.read()
                        b64 = base64.b64encode(data).decode()
                        href = f'<a href="data:image/png;base64,{b64}" download="watermarked_image.png" style="font-size:16px; padding:10px; background:#4CAF50; color: white; text-decoration:none; border-radius:5px;">📥 DOWNLOAD WATERMARKED IMAGE</a>'
                        display(HTML(href))
                    print("✅ Click the green button above ⬆️")
                except Exception as e:
                    print(f"❌ Error:  {e}")
            else:
                print("❌ Failed!")

        elif choice == '3':
            print("\n" + "="*50)
            print("3️⃣  VERIFY WATERMARK")
            print("="*50)

            img_path = upload_image("Upload WATERMARKED IMAGE:")
            if not img_path:
                continue

            logo_path = upload_image("Upload ORIGINAL LOGO:")
            if not logo_path:
                continue

            mode = input("\n⚙️  Mode [gpu/cpu]: ").strip().lower()
            if mode not in ['gpu', 'cpu']:
                mode = 'gpu'

            print(f"\n🔄 Verifying with MPI + {mode.upper()}...")
            cmd = ["mpirun", "--allow-run-as-root", "-np", "1", "bin/watermark", "check", img_path, logo_path, mode]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            ax1.imshow(Image.open(img_path))
            ax1.set_title('Watermarked', fontsize=11, fontweight='bold')
            ax1.axis('off')
            ax2.imshow(Image.open(logo_path))
            ax2.set_title('Original Logo', fontsize=11, fontweight='bold')
            ax2.axis('off')
            plt.tight_layout()
            plt.show()

        elif choice == '4':
            print("\n" + "="*50)
            print("4️⃣  EXTRACT WATERMARK")
            print("="*50)

            img_path = upload_image("Upload WATERMARKED IMAGE:")
            if not img_path:
                continue

            mode = input("\n⚙️  Mode [gpu/cpu]: ").strip().lower()
            if mode not in ['gpu', 'cpu']:
                mode = 'gpu'

            print(f"\n🔄 Extracting with MPI + {mode.upper()}...")
            cmd = ["mpirun", "--allow-run-as-root", "-np", "1", "bin/watermark", "extract", img_path, mode]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout)

            output_file = 'results/extracted_watermark.png'

            if os.path.exists(output_file):
                try:
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
                    ax1.imshow(Image.open(img_path))
                    ax1.set_title('Watermarked', fontsize=11, fontweight='bold')
                    ax1.axis('off')

                    ax2.imshow(Image.open(output_file), cmap='gray')
                    ax2.set_title('Extracted', fontsize=11, fontweight='bold')
                    ax2.axis('off')

                    plt.tight_layout()
                    plt.show()
                except Exception as e:
                    print(f"Display error: {e}")

                print("\n💾 Preparing download...")
                try:
                    with open(output_file, 'rb') as f:
                        data = f.read()
                        b64 = base64.b64encode(data).decode()
                        href = f'<a href="data:image/png;base64,{b64}" download="extracted_watermark. png" style="font-size: 16px; padding:10px; background:#2196F3; color:white; text-decoration:none; border-radius:5px;">📥 DOWNLOAD EXTRACTED WATERMARK</a>'
                        display(HTML(href))
                    print("✅ Click the blue button above ⬆️")
                except Exception as e:
                    print(f"❌ Error: {e}")
            else:
                print("❌ Failed!")

        elif choice == '5':
            print("\n👋 Goodbye!\n")
            break

        else:
            print("\n❌ Invalid!  Enter 1-5.")

        input("\n⏸️  [Press Enter]")

# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main function"""
    print("\n" + "="*60)
    print(" "*8 + "MPI + CUDA WATERMARKING SYSTEM")
    print(" "*10 + "For Google Colab")
    print("="*60 + "\n")

    setup_environment()
    create_source_files()

    if compile_code():
        print("\n✅ Ready!")
        run_interactive_menu()
    else:
        print("\n❌ Compilation failed!")

if __name__ == "__main__":
    main()


        MPI + CUDA WATERMARKING SYSTEM
          For Google Colab

🔧 SETTING UP ENVIRONMENT

📊 GPU Information:

📦 Installing dependencies...

📥 Downloading headers...
✅ Setup complete!

💾 CREATING SOURCE FILES (MPI + CUDA)
✅ MPI + CUDA source files created!

🔨 COMPILING MPI + CUDA
rm -rf build bin results
mkdir -p build bin results uploads
mpicc -O3 -Wall -Iinclude -c -o build/main.o src/main.c
gcc -O3 -Wall -Iinclude -c -o build/image_utils.o src/image_utils.c
nvcc -O3 -arch=sm_75 -Iinclude -c -o build/watermark_cuda.o src/watermark_cuda.cu

❌ Failed:
make: nvcc: No such file or directory
make: *** [Makefile:21: build/watermark_cuda.o] Error 127


❌ Compilation failed!
